# Demo: Writing Research Code with Claude

This notebook is a worked example of the pattern this starter is built around:

> **You make the decision. You point the AI at it. You review what it produces.**

It walks through a tiny end-to-end machine-learning study the same way you would on a real project, except the data is a small built-in dataset so the whole thing runs on your laptop in a few seconds. Nothing here touches Vista. The point is to *see* the loop, not to do real compute.

Each section shows three things:

1. **The decision** you made (in `templates/methodology.md`).
2. **The prompt** you would give Claude.
3. **The code Claude wrote**, which you then read and run.

When you do this for real, you would put your own data and method in `templates/methodology.md` and ask Claude to fill in `scripts/build_dataset.py`, `scripts/train_model.py`, and `scripts/evaluate.py`. This notebook just collapses those scripts into one place so you can read top to bottom.

---

## Setup

Everything below uses only what is already in `requirements.txt`. If you have not installed them yet:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility: one seed for the whole notebook.
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Resolve project paths relative to this notebook (notebooks/ -> project root).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = PROJECT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

# Colorblind-friendly palette, matching the project convention for figures.
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9"]

print("Project root:", PROJECT_ROOT)

## Stage 1-2 · The decision (what you put in `methodology.md`)

Before any code, you answer the five methodology questions. For this demo, pretend you filled them in like this:

| Question | Your answer |
|---|---|
| **1. What is the computation?** | A machine-learning regression model. |
| **2. What goes in?** | Each row is a California census block; predictors are demographics and housing stats. |
| **3. What comes out?** | A number: predicted median house value (in $100,000s). |
| **4. What is the method?** | A gradient-boosted tree, compared against a mean-predictor baseline. |
| **5. How do you know it worked?** | RMSE and R² on a held-out test set; beat the baseline; check which features drive predictions. |

That table is the whole knowledge base. Everything Claude writes below comes from it.

## Stage 3a · Build the dataset

**The prompt you'd give Claude:**

> *"Read `templates/methodology.md`. Implement `build_dataset.py`: load the data, engineer one feature that captures rooms-per-household, and return an analysis-ready table."*

**What Claude wrote** (here we load a built-in dataset instead of downloading real data, but the shape of the code is the same: load, engineer a feature, return a clean table):

In [ ]:
from sklearn.datasets import fetch_california_housing


def build_dataset():
    """Load raw data, engineer features, and return (X, y, feature_names).

    Mirrors what scripts/build_dataset.py would do: this is the only place
    feature engineering lives, so the train and evaluate steps stay simple.
    """
    raw = fetch_california_housing(as_frame=True)
    df = raw.frame.copy()

    # Engineered feature (the kind of thing that is often the publishable
    # contribution): average rooms per household, not just total rooms.
    df["rooms_per_household"] = df["AveRooms"] / df["AveOccup"]

    target = "MedHouseVal"
    feature_names = [c for c in df.columns if c != target]
    X = df[feature_names]
    y = df[target]
    return X, y, feature_names


X, y, feature_names = build_dataset()
print(f"{X.shape[0]:,} rows x {X.shape[1]} features")
X.head()

**Your job here:** read the code. Does `rooms_per_household` match what you meant? Is the join/clean step right? This is the *review* step. The AI proposes; you decide it is correct before moving on.

## Stage 3b · Train a baseline and the model

**The prompt:**

> *"Implement `train_model.py`: split train/test, train the baseline and the main model from the methodology, and save `results/metrics.json`."*

**What Claude wrote.** Note the baseline. A result only means something compared to something simpler, so methodology Question 4 always asks for a point of comparison. Here the baseline is "always predict the mean."

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Baseline: predict the training mean for every row.
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

# Main model: a gradient-boosted tree (the laptop-sized stand-in for the
# heavier model you would train on Vista via jobs/train.slurm).
model = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)

print("Trained baseline and gradient-boosted model.")

> **On the real project, this cell is the one that moves to HPC.** Training a full model with cross-validation and a hyperparameter sweep is exactly the heavy step that belongs on Vista. You would ask Claude (via the `tacc-vista` skill) to write `jobs/train.slurm`, submit it to the `gh` queue, and pull the saved model back. The laptop version here just lets you see the loop end to end.

## Stage 4 · Evaluate (did it beat the baseline?)

**The prompt:**

> *"Implement `evaluate.py`: compute RMSE and R² for both models on the test set and report whether the model beats the baseline."*

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score


def score(estimator, name):
    preds = estimator.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return {"model": name, "rmse": round(rmse, 3), "r2": round(r2_score(y_test, preds), 3)}


metrics = pd.DataFrame([
    score(baseline, "baseline (mean)"),
    score(model, "gradient boosting"),
])

improvement = 1 - metrics.loc[1, "rmse"] / metrics.loc[0, "rmse"]
print(f"Model cuts RMSE by {improvement:.0%} versus the baseline.\n")
metrics

## Stage 4 · Interpret (which features drive the prediction?)

A metric tells you *whether* the model works; interpretation tells you *why*, which is what reviewers want. The methodology lists SHAP. The cell below uses SHAP if it is installed and falls back to the model's built-in feature importances if it is not, so the notebook runs either way.

**The prompt:**

> *"Add SHAP to `evaluate.py` and save a feature-importance figure to `figures/` at 300 DPI with labeled axes."*

In [ ]:
try:
    import shap

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    importance = np.abs(shap_values).mean(axis=0)
    importance_label = "mean(|SHAP value|)"
    source = "SHAP"
except Exception as exc:  # shap not installed, or backend issue
    print(f"SHAP unavailable ({exc.__class__.__name__}); using model feature_importances_ instead.")
    importance = model.feature_importances_
    importance_label = "feature importance"
    source = "built-in importances"

ranked = (
    pd.DataFrame({"feature": feature_names, "importance": importance})
    .sort_values("importance", ascending=True)
)
print(f"Feature importance via {source}:")
ranked.iloc[::-1].reset_index(drop=True)

In [ ]:
# Publication-quality figure: 300 DPI, labeled axes, colorblind-friendly.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(ranked["feature"], ranked["importance"], color=PALETTE[0])
ax.set_xlabel(importance_label)
ax.set_ylabel("feature")
ax.set_title("What drives predicted house value")
fig.tight_layout()

out_path = FIGURES / "feature-importance-demo.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {out_path.relative_to(PROJECT_ROOT)}")
plt.show()

## What just happened

You ran a complete study: build → train → evaluate → interpret → figure. At every stage the shape was identical:

1. **You decided** the method in `methodology.md`.
2. **You prompted** Claude with a one-line ask pointing at that decision.
3. **You read** the code it wrote before running it.

Claude wrote the boilerplate (the train/test split, the metric functions, the SHAP fallback, the figure styling) so you spent your attention on the parts that are actually research: which feature to engineer, what baseline is fair, whether the result means anything.

### Taking this to a real project

- Put **your** data and method in `templates/methodology.md`.
- Ask Claude to fill in `scripts/build_dataset.py`, `scripts/train_model.py`, `scripts/evaluate.py` from it (those prompts are written at the top of each file).
- For the heavy training step, use the `tacc-vista` skill to write `jobs/train.slurm` and run it on Vista's `gh` queue, then pull results back here to visualize.

You bring the research; the scaffold, Claude, and HPC make it faster.